# Block 1: Your first API call from R

This notebook runs in Colab's **R** runtime (Runtime → Change runtime type → R).

**To keep your work, click *Copy to Drive* in the toolbar first.** Opened from the course link, the
notebook is not saved: your changes and outputs are lost when you close the tab. The copy goes to
*Colab Notebooks* in your Google Drive.

Two kinds of AI appear in this notebook:
- **The model you call from R** through the OpenRouter API (`or_chat()`): part of your analysis.
- **Gemini in Colab**, which helps you write and fix the code: your coding assistant.

Setup help (key, errors): `SETUP.md` in the course repository.

## Setup
Run the next three cells once, from top to bottom. Run them again after a runtime restart.

In [ ]:
# Packages: installs only what is missing
ensure_packages <- function(pkgs) {
  missing <- pkgs[!vapply(pkgs, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing) > 0) install.packages(missing)
  invisible(NULL)
}
ensure_packages(c("httr2", "jsonlite"))

In [ ]:
# Your OpenRouter key: paste it into the box and press Enter.
# The output is cleared right away so the key is not shown or saved.
Sys.setenv(OPENROUTER_API_KEY = readline("Paste your OpenRouter key: "))
IRdisplay::clear_output()
cat("Key set:", nzchar(Sys.getenv("OPENROUTER_API_KEY")))

In [ ]:
# Helper functions: or_chat() sends a prompt, parse_json_reply() reads JSON replies
# Helpers for calling an LLM through OpenRouter from R.
# Needs the packages httr2 and jsonlite, and your key in the environment
# variable OPENROUTER_API_KEY.

# Install only the packages that are missing (safe to run twice).
ensure_packages <- function(pkgs) {
  missing <- pkgs[!vapply(pkgs, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing) > 0) install.packages(missing)
  invisible(NULL)
}

# The OpenRouter key: from the environment, else from the Nuvolos space
# secret file (RStudio sessions on Nuvolos do not inherit the variable).
or_key <- function(secret_file = "/secrets/OPENROUTER_API_KEY") {
  key <- Sys.getenv("OPENROUTER_API_KEY")
  if (!nzchar(key) && file.exists(secret_file)) {
    key <- trimws(paste(readLines(secret_file, warn = FALSE), collapse = ""))
  }
  key
}

# Send one prompt to a model and return the reply as a character string.
or_chat <- function(prompt,
                    model = Sys.getenv("OPENROUTER_MODEL", "anthropic/claude-haiku-5.5"),
                    system = NULL,
                    temperature = 0,
                    max_retries = 3) {
  key <- or_key()
  if (!nzchar(key)) {
    stop("OPENROUTER_API_KEY is not set. Run the key cell first.", call. = FALSE)
  }

  messages <- list(list(role = "user", content = prompt))
  if (!is.null(system)) {
    messages <- c(list(list(role = "system", content = system)), messages)
  }

  resp <- httr2::request("https://openrouter.ai/api/v1/chat/completions") |>
    httr2::req_auth_bearer_token(key) |>
    httr2::req_body_json(list(model = model,
                              messages = messages,
                              temperature = temperature)) |>
    httr2::req_retry(max_tries = max_retries,
                     is_transient = \(r) httr2::resp_status(r) %in% c(429, 500, 502, 503)) |>
    httr2::req_error(body = \(r) {
      msg <- tryCatch(httr2::resp_body_json(r)$error$message,
                      error = \(e) NULL)
      paste0("HTTP ", httr2::resp_status(r), ": ",
             if (is.null(msg)) "no error message from OpenRouter" else msg)
    }) |>
    httr2::req_perform()

  httr2::resp_body_json(resp)$choices[[1]]$message$content
}

# Turn a model reply into an R object if it contains JSON.
# Strips ```json fences and surrounding prose; returns NULL with a warning
# when no JSON can be read.
parse_json_reply <- function(text) {
  text <- gsub("```(json)?", "", text)
  start <- regexpr("[\\[{]", text)
  if (start < 0) {
    warning("No JSON found in the reply.", call. = FALSE)
    return(NULL)
  }
  open <- substr(text, start, start)
  close <- if (open == "{") "}" else "]"
  ends <- gregexpr(close, text, fixed = TRUE)[[1]]
  candidate <- substr(text, start, max(ends))
  tryCatch(jsonlite::fromJSON(candidate),
           error = \(e) {
             warning("The reply contains text that is not valid JSON.", call. = FALSE)
             NULL
           })
}

## 1. Your first call

In [ ]:
reply <- or_chat("In one sentence: why might an economist call an LLM from R instead of using a chat window?")
cat(reply)

### Self-check

In [ ]:
stopifnot("The call returned no text. Is your key set?" = is.character(reply) && nchar(reply) > 0)
cat("Self-check passed.")

## 2. What `or_chat()` sends

`or_chat()` is a short function: it sends one HTTP request to OpenRouter and returns the text of
the reply. Here is the same request written out, so you can see everything that goes over the
wire and everything that comes back.

In [ ]:
resp <- httr2::request("https://openrouter.ai/api/v1/chat/completions") |>
  httr2::req_auth_bearer_token(or_key()) |>
  httr2::req_body_json(list(
    model = "anthropic/claude-haiku-5.5",
    messages = list(list(role = "user", content = "Name one use of GDP data. Five words at most.")),
    temperature = 0
  )) |>
  httr2::req_perform()

body <- httr2::resp_body_json(resp)
cat(body$choices[[1]]$message$content, "\n")
str(body$usage)   # tokens in and out: what you pay for

The reply is JSON: the text sits in `choices[[1]]$message$content`; `usage` counts the
tokens. Everything else in your analysis is ordinary R.

## 3. Roles: system and user

You saw system, user and assistant messages yesterday. In `or_chat()` the system message is an
argument. Same question, two system messages:

In [ ]:
q <- "Is a minimum wage increase good for workers?"
cat(or_chat(q, system = "You are a labour economist. Answer in two sentences, citing the main trade-off."), "\n\n")
cat(or_chat(q, system = "Answer only 'yes', 'no' or 'it depends'."))

## 4. Another model, same code

OpenRouter serves many models behind one API. Only the `model` argument changes.

In [ ]:
q <- "In one sentence: what does the Phillips curve describe?"
for (m in c("anthropic/claude-haiku-5.5", "deepseek/deepseek-v4.1-flash")) {
  cat(m, ":\n", or_chat(q, model = m), "\n\n", sep = "")
}

## 5. Temperature

At `temperature = 0` the model (almost always) picks its most likely answer. At higher values the
answers can vary. Run the cell twice: which answers change?

In [ ]:
for (t in c(0, 1)) {
  answers <- vapply(1:3, \(i) or_chat("Invent a name for a podcast about Swiss economics. Reply with the name only.", temperature = t), character(1))
  cat("temperature", t, ":", answers, "\n")
}

## 6. A reply you can compute with

For analysis you want data, not prose. Ask for JSON and turn it into an R object with
`parse_json_reply()`.

In [ ]:
reply <- or_chat(paste(
  "Classify the sentiment of each headline as positive, negative or neutral.",
  "Reply only with a JSON array of objects with the keys id and sentiment.",
  "1: Swiss exports hit record high",
  "2: Unemployment rises for third month",
  "3: Central bank leaves rates unchanged",
  sep = "\n"))
labels <- parse_json_reply(reply)
labels

### Self-check

In [ ]:
stopifnot("parse_json_reply() returned no table: look at `reply`" = is.data.frame(labels),
          "expected three rows, one per headline" = nrow(labels) == 3,
          "expected a column `sentiment`" = "sentiment" %in% names(labels))
cat("Self-check passed.")

## 7. Your turn, with Gemini

Now let Gemini write the code. Click **generate** in the empty cell below and ask, for example:

> Write R code that calls the function `or_chat(prompt)` once for each of the three questions in
> the character vector `questions` and stores the questions and answers in a data frame called
> `answers` with the columns `question` and `answer`.

Read the code before you run it. Does it call `or_chat()` once per question?

In [ ]:
questions <- c("What is inflation? One sentence.",
               "What is a recession? One sentence.",
               "What is GDP? One sentence.")

### Self-check

In [ ]:
stopifnot("Create the data frame `answers` first" = exists("answers") && is.data.frame(answers),
          "answers should have one row per question" = nrow(answers) == length(questions),
          "answers needs the columns question and answer" = all(c("question", "answer") %in% names(answers)))
cat("Self-check passed.")

## What this means for your project

Every model call in your project can be an R function call like these: you can repeat it, vary
the prompt, the model and the temperature, and keep the results in a table. That is what the
technical appendix of your project asks you to do.